# M1 regression · Colab 실행 진입점

로컬 소스 ZIP 업로드로 시작합니다. 최신 구현이 origin/main에 있다는 가정을 하지 않습니다. ZIP에는 데이터·checkpoint·.venv·.git을 넣지 않습니다. SOURCE_REVISION.txt의 commit SHA와 소스 hash를 결과에 기록합니다.

모든 기본 실행은 **synthetic_fixture 코드 검사**입니다. 실제 Colab/GPU 실행은 개발 세션에서 미검증입니다. 실제 데이터 학습은 관절·단위·좌표·calibration·split·RGB/pose 대응을 검증한 계약을 준비한 후 별도 단계로 진행합니다. 원본 압축 데이터 준비는 이 notebook의 기능이 아닙니다.

In [ ]:
import os, sys, re, shutil
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
try:
    import torch
except ImportError:
    raise RuntimeError("PyTorch가 필요합니다. 공식 CUDA 설치법을 확인하고 GPU 환경을 CPU wheel로 덮어쓰지 마세요.")
print({"python": sys.version.split()[0], "torch": str(torch.__version__), "cuda_available": torch.cuda.is_available(),
       "free_disk_bytes": shutil.disk_usage(".").free})
version = re.match(r"^(\d+)\.(\d+)", str(torch.__version__))
if not version or tuple(map(int, version.groups())) < (2, 3) or not callable(getattr(getattr(torch, "amp", None), "GradScaler", None)):
    raise RuntimeError("PyTorch >=2.3와 torch.amp.GradScaler API가 필요합니다. 기존 GPU PyTorch를 자동 교체하지 않습니다.")
print("로컬 검증: Python 3.12.14 / torch 2.7.1+cpu. 다른 버전에서는 아래 테스트를 먼저 통과해야 합니다.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print({"gpu": p.name, "total_vram_bytes": p.total_memory, "free_vram_bytes": torch.cuda.mem_get_info()[0]})


## 소스 ZIP 업로드

사용자가 로컬 소스 ZIP을 선택합니다. 이 셀은 소스 코드 ZIP만 처리하며 데이터 archive를 결합·추출하지 않습니다. 압축 크기 64MiB·해제 크기 128MiB 상한을 두고, 데이터·환경·Git 폴더를 제외합니다.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, stat, tempfile, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("소스 ZIP 한 개를 선택하세요.")
zip_name, zip_bytes = next(iter(uploaded.items()))
if not zip_name.endswith(".zip") or len(zip_bytes) > 64*1024**2:
    raise ValueError("64MiB 이하의 소스 ZIP 필요")
extract_root = Path(tempfile.mkdtemp(prefix="rewind_source_", dir="/content"))
excluded = {".git", ".venv", "data", "outputs", "checkpoints", ".aws", ".codex", ".agents"}
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    members = archive.infolist()
    if sum(m.file_size for m in members) > 128*1024**2:
        raise ValueError("소스 ZIP의 해제 크기 상한 초과")
    for member in members:
        p = PurePosixPath(member.filename)
        if p.is_absolute() or ".." in p.parts or any(":" in part for part in p.parts) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("잘못된 ZIP member 경로")
        if not any(part in excluded for part in p.parts):
            archive.extract(member, extract_root)
roots = [p.parent.parent for p in extract_root.rglob("rewind_m1/__init__.py")]
if len(roots) != 1:
    raise ValueError("rewind_m1 패키지가 하나인 프로젝트 ZIP 필요")
PROJECT_ROOT = roots[0]
print("source:", PROJECT_ROOT)
stamp = PROJECT_ROOT / "SOURCE_REVISION.txt"
print("revision:", stamp.read_text().strip() if stamp.exists() else "unversioned; code hash로 기록")

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_ROOT, check=True)

## 결과 경로

학습은 임시 디스크에서 수행합니다. 런타임 삭제 시 결과가 소실될 수 있으므로 종료 전에 마지막 export 셀을 실행하세요. 영구 보관하려면 PERSIST_TO_DRIVE를 True로 바꾸고 사용자가 Colab의 Drive 인증을 완료합니다. 인증키를 코드에 저장하지 않습니다. Export는 원본 archive와 다른 실험 디렉터리에 결과를 복사하고 파일별 SHA-256을 다시 읽어 확인합니다. 실제 Drive 실행은 미검증입니다.


In [ ]:
import datetime, uuid
PERSIST_TO_DRIVE = False
run_name = "m1_synthetic_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_ROOT = PROJECT_ROOT / "outputs" / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=False)
EXPORT_ROOT = None
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    EXPORT_ROOT = Path("/content/drive/MyDrive/REWIND_runs") / run_name
print("임시 학습 경로:", RUN_ROOT, "; data_kind=synthetic_fixture")


In [ ]:
import json
for model_name in ("single", "temporal"):
    subprocess.run([sys.executable, "-m", "rewind_m1", "train",
        "--config", f"configs/m1_{model_name}_synthetic.json", "--output", str(RUN_ROOT/model_name),
        "--steps", "80", "--device", DEVICE], cwd=PROJECT_ROOT, check=True)
    report = json.loads((RUN_ROOT/model_name/"report.json").read_text())
    print(model_name, report["final_train"], report["validation"])

In [ ]:
subprocess.run([sys.executable, "-m", "rewind_m1", "train",
    "--config", "configs/m1_single_synthetic.json", "--output", str(RUN_ROOT/"single"),
    "--steps", "100", "--resume", str(RUN_ROOT/"single"/"last.pt"), "--device", DEVICE],
    cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, "-m", "rewind_m1", "evaluate",
    "--checkpoint", str(RUN_ROOT/"single"/"last.pt"), "--split", "validation",
    "--output", str(RUN_ROOT/"single"/"evaluation.json"), "--device", DEVICE],
    cwd=PROJECT_ROOT, check=True)

## 개발용 2D front-end·예측 캐시

작은 Conv heatmap 모델의 CPU 합성 gradient 진단입니다. 로드맵의 EfficientNet front-end 재현이나 실제 RGB 평가를 완료한 결과가 아닙니다. Confidence는 heatmap peak mass이며 visibility는 unknown으로 유지합니다.

In [ ]:
subprocess.run([sys.executable, "-m", "rewind_m1.frontend",
    "--output", str(RUN_ROOT/"frontend")], cwd=PROJECT_ROOT, check=True)
print(json.loads((RUN_ROOT/"frontend"/"report.json").read_text()))

## Train-only front-end → 고정 예측 캐시 → regression

합성 stereo RGB 24개 중 train 12개만 front-end 학습에 사용합니다. 고정된 모델의 예측 2D를 sequence/frame으로 GT 3D와 연결하고 regression을 실행합니다. Validation RGB/label은 front-end optimizer에 사용하지 않습니다. `frontend_weights.pt`는 버전·모델 설정·fingerprint를 포함한 weights-only 파일이며 front-end 학습 재개용 optimizer/RNG 파일이 아닙니다. Confidence는 visibility 정답으로 사용하지 않습니다. 이 작은 CPU 경로는 실제 데이터 baseline과 구분합니다.


In [ ]:
subprocess.run([sys.executable, "-m", "rewind_m1.predicted_fixture",
    "--config", "configs/m1_single_synthetic.json", "--output", str(RUN_ROOT/"predicted"),
    "--frontend-steps", "20", "--steps", "30"], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, "-m", "rewind_m1", "evaluate",
    "--checkpoint", str(RUN_ROOT/"predicted"/"last.pt"), "--dataset", str(RUN_ROOT/"predicted"/"features.json"),
    "--output", str(RUN_ROOT/"predicted"/"evaluation.json"), "--device", "cpu"], cwd=PROJECT_ROOT, check=True)


## 완료 결과 영구 보관 (선택)

이 셀은 현재 완료된 checkpoint·설정·manifest·로그를 복사합니다. 학습 중 자동 Drive 동기화는 구현하지 않았습니다. 런타임이 갑자기 종료되면 마지막 export 이후 결과를 잃을 수 있습니다. 실패 시 부분 복사 경로를 명시하며 원본 학습 결과는 임시 디스크에 유지합니다.


In [ ]:
import hashlib
def export_results(source, destination):
    if destination.exists():
        raise ValueError("기존 export를 덮어쓰지 않습니다. 새 실행 디렉터리를 선택하세요.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copytree(source, destination)
        checks = {}
        for original in sorted(source.rglob("*")):
            if not original.is_file():
                continue
            relative = original.relative_to(source)
            expected = hashlib.sha256(original.read_bytes()).hexdigest()
            copied = destination / relative
            if hashlib.sha256(copied.read_bytes()).hexdigest() != expected:
                raise IOError(f"Export 읽기 확인 실패: {relative}")
            checks[relative.as_posix()] = expected
        (destination/"export_manifest.json").write_text(json.dumps({"source_run":source.name,"file_sha256":checks},indent=2)+"\n",encoding="utf-8")
        print("복사·파일별 hash 확인:", destination, len(checks), "files")
    except Exception:
        print("부분 export를 확인하세요:", destination)
        raise
if EXPORT_ROOT is not None:
    export_results(RUN_ROOT, EXPORT_ROOT)
else:
    print("임시 디스크 결과만 존재합니다:", RUN_ROOT)


## 다음 단계

현재 결과는 인위적 fixture의 코드 검증입니다. 실제 split·좌표·단위·calibration·좌우 RGB/pose/frame 대응을 검증한 m1_features_v1 계약, 실제 RGB 로더와 front-end 학습/캐시 연결, Colab GPU·VRAM·준비 시간·학습 비용 검증이 필요합니다. M1 전체 완료 및 실제 정확도·30 FPS를 주장하지 않습니다.

실제 데이터 CLI는 검증된 계약 없이는 거부합니다. Test split은 튜닝에 사용하지 않으며 최종 평가에만 명시적 --final-test로 사용합니다. 상세 계약은 docs/M1_BASELINE.md, Colab 환경 안내는 docs/COLAB_SETUP.md를 참조하세요.